In [ ]:
%pip install -Uq langchain langchain-openai langchain-community langchain-text-splitters

# Map-Reduce 模式（分而治之）

In [13]:
from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Get API Key
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# 1. 初始化模型
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, openai_api_key=OPENAI_API_KEY)

# 2. 讀取長文本文件
with open("/content/drive/MyDrive/Colab Notebooks/LangChain Research/long_content_example.txt", "r", encoding="utf-8") as f:
    txt_raw_data = f.read()

# 3. 建立文本分割器並切割文件
text_splitter = TokenTextSplitter(chunk_size=3000, chunk_overlap=100)
chunks = text_splitter.split_text(txt_raw_data)

print(f"文件被分割成了 {len(chunks)} 個區塊。")

文件被分割成了 5 個區塊。


In [22]:
# 4. Map：每個 chunk 先做摘要
# 這是 sequential 處理的寫法，沒有碰觸到 map-reduce 的平行處理精神
# 透過 %%time 得知耗時 Wall time: 9.79 s (5個chunks)

%%time

map_prompt = ChatPromptTemplate.from_template(
    "你是一個嚴謹的摘要助手。請用繁體中文摘要下面內容，保留關鍵名詞與數字，不要加入未提及的資訊。\n\n{text}"
)

map_chain = map_prompt | llm | StrOutputParser()

chunk_summaries = [map_chain.invoke({"text": c}) for c in chunks]

CPU times: user 32.1 ms, sys: 930 µs, total: 33 ms
Wall time: 9.79 s


In [23]:
# 5. Reduce：把所有 chunk 摘要再彙總成最終摘要

%%time

reduce_prompt = ChatPromptTemplate.from_template(
    "以下是一系列分段摘要。請將它們整合成一份最終摘要（繁體中文），要求：\n"
    "1) 去重、合併同義內容\n"
    "2) 結構清楚（分段即可，不用很花）\n"
    "3) 使用標題與項目符號\n"
    "4) 保留重要名詞、數字、因果關係\n\n"
    "{summaries}"
)

reduce_chain = reduce_prompt | llm | StrOutputParser()

final_summary_text = reduce_chain.invoke({"summaries": "\n\n".join(chunk_summaries)})

print("\n--- 最終摘要 ---")
print(final_summary_text)


--- 最終摘要 ---

## Claude 插件功能全解析

- Anthropic 推出的最新 Claude 插件功能被指為導致美股暴跌的原因之一
- Claude Desktop 的 Cowork 功能提供各種內置插件，如 bioresearch、customer support、data整理、enterprise search、finance、legal、marketing、product management 和 productivity
- 用戶可以瀏覽、上傳自己的插件，或在 GitHub Marketplace 上尋找喜歡的插件
- 插件功能可以透過斜杠命令執行各種任務，並連接各種軟件或外部系統，提高工作效率

## 多功能軟件應用

- 可連接任何軟件或外部MCP系統，應用在法律、財務、銷售行銷等多個領域
- 包括效率引擎、AI銷售全流程優化和AI客服自動化處理流程等功能
- 自动归档并储存对话在知识库中，可进行人工审核后发送或直接自动回复客户
- 产品经理功能包括自动撰写PRD和规划路线图，透明化项目规格
- 市场营销部分可撰写专案简报和自动化品牌审查，制定SEO内容和日历规划
- AI法律风险分析可上传合约后自动分析核心条款和风险，避免高额律师费
- 财务和数据分析驱动，一键提取资料库数据，提高效率

## 強大且靈活的功能

- 可從資料庫提取資料、進行對賬、財務表生成、現金流分析和預測，提高效率
- 企業搜索功能整合不同工具中的信息，提供簡單答案
- 生物科研神器過濾網絡上的垃圾信息，搜尋權威學術文獻
- Claude 的 Plugins 包含 command、Connector 和 skills，可連接不同工具
- 用戶可以客制化 skills，根據公司情況編輯
- 提供技能指南供免費下载，作者也提供VIP群和课程
- 重要的是如何驾驭和使用工具，以提高业务效率，作者鼓勵观众尝试 Claude
- 在AI时代，工具可以提高工作效率，但不能完全取代专业人员，重要的是如何使用工具
- 作者希望观众订阅频道並點贊支持。
CPU times: user 14.4 ms, sys: 1.94 ms, total: 16.4 ms
Wall time: 7.58 s


## 真正 Map 平行寫法

In [21]:
# 真正 Map 平行寫法 -- 用 batch()
# 這樣會同時送多個 request 給 model（受 rate limit 限制）。
# 透過 %%time 得知耗時 Wall time: 2.12 s
# 已知有5個 chunks，確實約 4.6 倍提升

%%time

chunk_summaries = map_chain.batch(
    [{"text": c} for c in chunks],
    config={"max_concurrency": 5}
)


CPU times: user 64.4 ms, sys: 3.5 ms, total: 67.9 ms
Wall time: 2.12 s


## Prompt 是 LCEL Pipeline 的核心

在 LCEL 的架構中，**Prompt 就是這條生產線的「軟體定義（Software Defined）」層**。

過去使用 `load_summarize_chain` 時，你像是買了一台「自動果汁機」，按個鈕就出結果；而現在用 `reduce_prompt | llm | StrOutputParser()`，你是在「設計生產流程」。

這正是體現你身為開發者「思辨力」的地方。針對 Map-Reduce 中的 **Reduce 階段**，Prompt 的寫法會直接決定最終摘要的品質。

---

### 1. 為什麼 Prompt 在 Reduce 階段至關重要？

在 Reduce 階段，模型拿到的不是原始長文，而是**一堆散落的小摘要**。如果 Prompt 寫得不好，會產生以下問題：

* **資訊重複**：模型只是把五個小摘要接在一起，沒有邏輯整合。
* **幻覺（Hallucination）**：模型試圖填補小摘要之間的空白，結果編造了原文沒說的事。
* **失去脈絡**：忽略了故事的起承轉合，只剩下零碎的事實。

---

### 2. 兩種不同思維的 Prompt 寫法對比

假設你正在處理電商市場調研報告：

#### A. 平庸的寫法（僅做彙整）

> 「以下是幾份報告的摘要，請將它們整合在一起。」

* **結果**：模型會產出一份長度增加，但洞察力為零的文字堆砌。

#### B. 具備「分析與邏輯」的寫法（LCEL 的真正價值）

> 「你是一位資深的商業分析師。以下是從長篇調研中提取的多個片段摘要。
> 請執行以下邏輯：
> 1. **辨識核心趨勢**：找出所有摘要中共同提到的三個關鍵點。
> 2. **發掘矛盾點**：若不同摘要對同一個數據有衝突，請特別標註。
> 3. **結構化輸出**：先給出一個 50 字的執行摘要，再以條列式呈現各維度洞察。
>
>
> 摘要內容如下：
> {docs_content}」

---

### 3. 質疑前提：LCEL 一定比較好嗎？

雖然 LCEL 提供了極大的自由度，但我們也要驗證一個邏輯：**「如果你追求的是極速開發，LCEL 其實增加了複雜度。」**

* **優勢**：當你的商業邏輯（例如跨境電商的競品分析）需要特定的輸出格式（JSON、Markdown 表格）時，LCEL 的 Prompt 是唯一的救星。
* **挑戰**：你需要自己處理 `token` 溢出的問題。如果 Reduce 階段的摘要加起來還是太長，超過了 `gpt-3.5-turbo` 的限制，這條 Pipeline 就會報錯。這時你可能需要設計「遞迴式（Recursive）」的 Reduce。


# Refine（迭代完善）

## Step 0：安裝函式庫

In [ ]:
%pip install -Uq wikipedia pypdf docx2txt

## Step 1：初始化模型

In [8]:
# LangChain 1.x 最新版 runnable 寫法
# 核心概念保留：先「初步摘要」，再用「目前摘要 + 下一份文件」逐步 refine，並保留 intermediate_steps

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.document_loaders import WikipediaLoader, PyPDFLoader, Docx2txtLoader

# Get API Key
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# -----------------------------
# 1) 初始化模型
# -----------------------------
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0, openai_api_key=OPENAI_API_KEY)
parser = StrOutputParser()


## Step 2：載入不同來源文件

In [10]:
# -----------------------------
# 2) 載入不同來源文件（示範）
#    - WikipediaLoader: 線上資料
#    - PyPDFLoader: 本地 PDF
#    - Docx2txtLoader: 本地 Word
# -----------------------------

# Wikipedia
wiki_loader = WikipediaLoader(query="工廠方法模式", load_max_docs=1, lang="zh")
wiki_docs = wiki_loader.load()

# PDF
pdf_loader = PyPDFLoader("/content/drive/MyDrive/Colab Notebooks/LangChain Research/factory_pattern.pdf")
pdf_docs = pdf_loader.load()

# Word
word_loader = Docx2txtLoader("/content/drive/MyDrive/Colab Notebooks/LangChain Research/simple_factory_vs_factory_method.docx")
word_docs = word_loader.load()

# 合併所有來源
all_docs = wiki_docs + pdf_docs + word_docs
print(f"總共載入了 {len(all_docs)} 份文件區塊。")

總共載入了 3 份文件區塊。


## Step 3：定義 Refine 核心 prompt

In [35]:
# -----------------------------
# 3) 定義 Refine 核心：兩個 prompt
#    - 初步摘要：只看第一份文件，產出 initial summary
#    - 迭代完善：把「目前摘要」+「下一份文件」一起丟給 LLM，要求更新並保留連貫性
# -----------------------------
initial_prompt = ChatPromptTemplate.from_template(
    "你是一個嚴謹的摘要助手。請用繁體中文對以下內容做『初步摘要』。\n"
    "要求：保留重要名詞、數字、因果關係；不要加入不存在的資訊。\n\n"
    "內容：\n{document_text}"
)

refine_prompt = ChatPromptTemplate.from_template(
    "你是一個嚴謹的摘要助手。以下有一份『目前摘要』與『新文件內容』。\n"
    "請根據新文件內容，『更新並完善』目前摘要，產出更完整且連貫的摘要。\n\n"
    "硬性要求：\n"
    "1) 若新文件提供新事實/數字/例子，必須吸收進摘要\n"
    "2) 若新文件與現有摘要有衝突，請以更可信/更具體者為準，並用一句話標註存在差異\n"
    "3) 去重、合併同義內容\n"
    "4) 使用標題與項目符號\n"
    "5) 不要捏造、不要腦補\n\n"
    "目前摘要：\n{current_summary}\n\n"
    "新文件內容：\n{document_text}\n\n"
    "請輸出『更新後摘要』（繁體中文）："
)

initial_chain = initial_prompt | llm | parser
refine_chain = refine_prompt | llm | parser

## Step 4：執行 Refine

In [36]:
# -----------------------------
# 4) 執行 Refine（保留 intermediate_steps）
# -----------------------------
def refine_summarize(docs):
    """
    docs: list[Document] (langchain_core.documents.Document)
    return:
      final_summary: str
      intermediate_steps: list[str]  # 每一步更新後的摘要
    """
    if not docs:
        return "", []

    # 取出第一份文件做初步摘要
    first_text = docs[0].page_content
    current_summary = initial_chain.invoke({"document_text": first_text})
    intermediate_steps = [current_summary]

    # 逐份文件迭代完善（Refine 本質上是序列化，強調連貫性）
    for i, d in enumerate(docs[1:], start=2):
        doc_text = d.page_content
        current_summary = refine_chain.invoke(
            {"current_summary": current_summary, "document_text": doc_text}
        )
        intermediate_steps.append(current_summary)

    return current_summary, intermediate_steps


final_summary, intermediate_steps = refine_summarize(all_docs)

print("\n--- 每一次的迭代摘要 ---")
for i, step in enumerate(intermediate_steps, start=1):
    print(f"\n【第 {i} 步摘要】\n{step}\n")

print("\n--- 最終完善摘要 ---")
print(final_summary)


--- 每一次的迭代摘要 ---

【第 1 步摘要】
初步摘要：

面向對象程序設計中，工廠是一種用來創建其他對象的抽象構造方法，實現不同的分配方案。工廠對象包含一個或多個方法，根據參數創建並返回相應的對象。當對象控制過程複雜時，工廠可動態創建對象、從對象池返回對象，或對對象進行配置。多種設計模式（如工廠方法模式、抽象工廠模式、建造者模式、單例模式）均應用工廠概念。

簡單工廠模式中，單一工廠類根據參數選擇具體對象，工廠方法可設為靜態方法，封裝複雜創建過程，如根據圖像文件格式選擇對應的ImageReader類。

工廠方法模式允許子類決定創建對象的具體類型，也可不依賴多態性使用工廠方法以獲得其他好處。工廠方法通常有描述性名稱，避免構造方法重載的歧義，且構造方法可設為私有，強制使用工廠方法創建對象。

示例部分展示了Java、VB.NET、C#中復數對象的工廠方法實現。

工廠方法存在三大局限：一是重構已有類（如將構造方法設為私有）會破壞現有客戶端代碼；二是私有構造方法阻止類的擴展，因子類無法調用父類私有構造方法；三是子類需實現所有工廠方法，否則返回父類實例，部分語言可用反射避免此問題。修改底層語言使工廠方法成為第一類成員可緩解這些局限。


【第 2 步摘要】
更新後摘要

工廠模式概述
- 工廠模式的核心在於「將物件的建立邏輯封裝起來」，避免程式碼中直接使用 new 來實例化特定類別，從而實現解耦（Decoupling），讓呼叫方不需了解物件產生的細節。
- 透過工廠模式，可以遵守軟體設計的「開放封閉原則」，即對擴充開放、對修改封閉，避免在新增產品時修改大量使用 new 的程式碼。

工廠模式的三種層次
1. 簡單工廠（Simple Factory）
   - 非正式設計模式，由單一類別根據傳入參數（如字串 "Cheese"）決定要 new 哪一種產品。
2. 工廠方法（Factory Method）
   - 定義一個建立物件的介面（Interface），由子類別決定實例化哪個具體類別，實例化延遲到子類別實作時。
3. 抽象工廠（Abstract Factory）
   - 提供一個介面用於建立「相關或依賴物件的家族」，不需指定具體類別，例如同時產生「Mac 風格的按鈕」與「Mac 風格的捲軸」。

工廠模式的應用與價值
- 工廠模式廣泛應用於面向對象程序設

## RAG 架構的最大公因數


但我會把它講得更精準一點：**Refine chain 不是 RAG 的最大公因數**；它更像是「把資訊逐步吸收、維持連貫」的**一種彙整策略**。而 RAG 的最大公因數其實是「檢索增強」那條主幹：**Retrieve → Ground → Generate**。

你可以這樣在腦中分層，會非常清楚：

第一層：RAG 的最大公因數（幾乎所有 RAG 都長這樣）

1. 把資料變成可檢索的形狀（chunk/embedding/index）
2. 針對問題去檢索（retrieve top-k + rerank 可選）
3. 把檢索到的內容當作上下文，要求模型回答（grounded generation）

用一句話就是：**先找得到，再基於找到的回答。**

第二層：Refine 是「處理上下文」的一種策略
Refine 的強項是：當你有一串文件（或 chunk）要整合成一個連貫的輸出時，用「目前摘要 + 新資料」一直更新，避免 Map-Reduce 那種「先分散總結再合併」導致跨文件脈絡流失。

所以更準確的關係是：
**Refine 可以放在 RAG 的某一段當作 aggregator / synthesizer**，例如：

情境 A：你問一個問題，retrieve 出來很多段落

* 你可以用 Map-Reduce 做「多段摘要後整合」
* 也可以用 Refine 做「一段一段吃進來，把答案越修越完整」

情境 B：你不是回答問題，而是要做「多來源報告整合」

* Refine 更像主角（因為目標是高品質綜述）

一個很具體的例子（讓你有畫面）：
你要做「工廠方法模式」的研究整理。你有 Wikipedia + PDF 教材 + Word 對照文章。
Refine 的做法像是你先讀 Wikipedia 寫初稿，接著讀 PDF 修正術語、補上例子，最後讀 Word 把對比表與常見誤解補進去。最後得到的是一份「連貫、逐步打磨」的內容。

而 RAG 的做法更像：
你問「工廠方法模式跟簡單工廠差在哪？」→ 系統先去資料庫找最相關段落 → 把段落貼給模型 → 模型只針對問題回答。

所以你的判斷「在這個架構下再去針對不同情境做延伸」是對的，但要把“最大公因數”那句換一下會更準：

更準的一句話：
**Refine 是很多知識整合/長文理解/多文件綜述的最大公因數；而 RAG 的最大公因數是 Retrieve→Ground→Generate，Refine 只是其中一種整合（synthesis）方法。**

你接下來要延伸的方向，我會給你三個最常見的分支（也最實用）：

1. RAG 問答型：retrieve 出來的多段 context 用 Refine 逐段吸收，產生「更像人寫的答案」
2. 報告整合型：Refine + intermediate_steps，直接變成「可追溯的寫作過程」
3. Decision / Agent 型：Refine 不是最後輸出，而是把摘要餵給下一段「決策提示 + tool call」

如果你願意，我可以用你現在這份「工廠方法模式」資料當例子，畫出你正在走的 pipeline：

ingest（多來源）→ normalize → chunk → （可選：retrieve）→ refine synthesize → final report / QA / decision

你會很快看懂你接下來該把「retrieve」插在哪裡，以及什麼時候用 Refine、什麼時候用 Map-Reduce。
